# Schrödinger's Cat Lab

## Introduction

In 1935, Erwin Schrödinger proposed a famous thought experiment to illustrate the strange implications of quantum mechanics. A cat is placed in a sealed box along with a radioactive atom, a Geiger counter, and a vial of poison. If the atom decays, the Geiger counter triggers and releases the poison. Until we open the box and look, the atom—and thus the cat—exists in a **quantum superposition** of states.

### Key Concepts

- **Quantum superposition**: A system can exist in a combination of states before measurement, described by a wave function $|\psi\rangle$.
- **Measurement collapse**: When we measure, the system "collapses" to one definite outcome. The act of observation changes the system.
- **Born rule**: The probability of each outcome is given by $|\alpha|^2$ for the corresponding amplitude $\alpha$. This is the statistical interpretation of quantum mechanics.

In this lab, we model the cat as a simple **two-state quantum system**. You can prepare states, perform measurements, and run statistical experiments to see how probabilities emerge from repeated trials.

## Imports

In [3]:
import sys

PROJECT_ROOT = "/Users/owner/Desktop/Educational_Projects/schrodinger-cat-lab"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from quantum_cat.state import prepare_state, get_probabilities
from quantum_cat.measurement import measure_state
from quantum_cat.experiments import run_trials, count_outcomes
from quantum_cat.visuals import draw_cat_box, plot_probabilities, plot_histogram

#from quantum_cat import (
    #prepare_state,
    #get_probabilities,
    #measure_state,
    #run_trials,
    #count_outcomes,
    #draw_cat_box,
    #plot_probabilities,
    #plot_histogram,
#)

## Quantum Model

We represent the cat's state as a **quantum state vector**:

$$|\psi\rangle = \alpha |alive\rangle + \beta |dead\rangle$$

with normalization $|\alpha|^2 + |\beta|^2 = 1$.

The **Born rule** gives the measurement probabilities:

$$P(alive) = |\alpha|^2$$

$$P(dead) = |\beta|^2$$

A measurement randomly selects an outcome according to these probabilities and **collapses** the state to that basis vector.

## Widget Controls

In [5]:
amplitude_slider = widgets.FloatSlider(
    value=0.707,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Alive amp:',
    style={'description_width': 'initial'},
    readout_format='.2f',
)

phase_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=2 * np.pi,
    step=0.1,
    description='Phase (rad):',
    style={'description_width': 'initial'},
    readout_format='.2f',
)

trials_slider = widgets.IntSlider(
    value=100,
    min=1,
    max=1000,
    step=10,
    description='Trials:',
    style={'description_width': 'initial'},
)

prepare_btn = widgets.Button(description='Prepare State', button_style='primary')
measure_btn = widgets.Button(description='Measure')
run_trials_btn = widgets.Button(description='Run Trials')
reset_btn = widgets.Button(description='Reset')

## App State

In [6]:
app_state = {
    "psi": None,
    "probabilities": None,
    "measured_outcome": None,
    "trial_results": [],
}

## Output Panels

In [7]:
visual_output = widgets.Output()
probability_output = widgets.Output()
histogram_output = widgets.Output()
state_output = widgets.Output()

## Render Functions

In [8]:
def render_visual():
    with visual_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5), facecolor='#0a0a0f')
        probs = app_state["probabilities"] or {"alive": 0.5, "dead": 0.5}
        draw_cat_box(ax, probs, app_state["measured_outcome"])
        plt.tight_layout()
        plt.show()

def render_probabilities():
    with probability_output:
        clear_output(wait=True)
        probs = app_state["probabilities"]
        if probs is None:
            return
        fig, ax = plt.subplots(figsize=(3, 3), facecolor='#0a0a0f')
        plot_probabilities(ax, probs)
        plt.tight_layout()
        plt.show()

def render_histogram():
    with histogram_output:
        clear_output(wait=True)
        results = app_state["trial_results"]
        if not results:
            return
        counts = count_outcomes(results)
        fig, ax = plt.subplots(figsize=(3, 3), facecolor='#0a0a0f')
        plot_histogram(ax, counts)
        plt.tight_layout()
        plt.show()

def render_state_text():
    with state_output:
        clear_output(wait=True)
        psi = app_state["psi"]
        outcome = app_state["measured_outcome"]
        results = app_state["trial_results"]
        if psi is None:
            print("No state prepared. Click 'Prepare State'.")
            return
        a, b = psi[0], psi[1]
        print(f"|ψ⟩ = {a.real:.3f}{'+' if a.imag >= 0 else ''}{a.imag:.3f}i |alive⟩ + {b.real:.3f}{'+' if b.imag >= 0 else ''}{b.imag:.3f}i |dead⟩")
        if outcome:
            print(f"\nMeasured: {outcome}")
        if results:
            n = len(results)
            c = count_outcomes(results)
            print(f"\nTrials: {n}")
            print(f"  alive: {c['alive']} ({100*c['alive']/n:.1f}%)")
            print(f"  dead:  {c['dead']} ({100*c['dead']/n:.1f}%)")

## Callback Functions

In [9]:
def on_prepare_clicked(b):
    a = amplitude_slider.value
    phi = phase_slider.value
    app_state["psi"] = prepare_state(a, phi)
    app_state["probabilities"] = get_probabilities(app_state["psi"])
    app_state["measured_outcome"] = None
    app_state["trial_results"] = []
    render_visual()
    render_probabilities()
    render_histogram()
    render_state_text()

def on_measure_clicked(b):
    psi = app_state["psi"]
    if psi is None:
        return
    outcome, collapsed = measure_state(psi)
    app_state["measured_outcome"] = outcome
    app_state["psi"] = collapsed
    app_state["probabilities"] = get_probabilities(collapsed)
    render_visual()
    render_probabilities()
    render_state_text()

def on_run_trials_clicked(b):
    psi = app_state["psi"]
    if psi is None:
        return
    n = trials_slider.value
    app_state["trial_results"] = run_trials(psi, n)
    render_histogram()
    render_state_text()

def on_reset_clicked(b):
    a = amplitude_slider.value
    phi = phase_slider.value
    app_state["psi"] = prepare_state(a, phi)
    app_state["probabilities"] = get_probabilities(app_state["psi"])
    app_state["measured_outcome"] = None
    app_state["trial_results"] = []
    render_visual()
    render_probabilities()
    render_histogram()
    render_state_text()

prepare_btn.on_click(on_prepare_clicked)
measure_btn.on_click(on_measure_clicked)
run_trials_btn.on_click(on_run_trials_clicked)
reset_btn.on_click(on_reset_clicked)

## Layout

In [10]:
controls = widgets.VBox([
    amplitude_slider,
    phase_slider,
    trials_slider,
    widgets.HBox([prepare_btn, measure_btn, run_trials_btn, reset_btn]),
])

right_panel = widgets.VBox([
    probability_output,
    histogram_output,
])

main_layout = widgets.HBox([
    controls,
    visual_output,
    right_panel,
])

display(main_layout)
display(state_output)

Output()

## Exploration Prompts

Try these activities to deepen your intuition:

1. **Balanced superposition**: Set the alive amplitude to $1/\sqrt{2} \approx 0.707$ (or use phase 0). Prepare the state and run 500 trials. Compare the empirical frequencies with the theoretical probabilities (50% each).

2. **Asymmetric states**: Change the amplitude and observe how the histogram shifts. With amplitude 0.9, you should see roughly 81% alive and 19% dead.

3. **Measurement collapse**: Prepare a superposition, measure once, then click **Reset** to return to the superposition. Notice how the visual changes from uncertain (two overlapping silhouettes) to definite (one clear outcome) after measurement.

4. **Phase exploration**: The phase $\phi$ affects the relative phase between $|alive\rangle$ and $|dead\rangle$. For a two-outcome measurement in the alive/dead basis, the probabilities depend only on $|\alpha|^2$ and $|\beta|^2$, so the phase does not change the outcome distribution. It matters for other measurements (e.g., in a rotated basis), which we leave for future versions.